# Le topic modeling avec BERTopic

Une approche de ML, non surpervisée, permettant d'extraire les thématiques/topics présents dans un ensemble de données textuelles.

**Ressources :**  
- le très bon tuto de Axel Morin : https://css-polytechnique.github.io/css-ipp-materials/pages/bertopic-tutorial.html
- le site de BERTopic : https://maartengr.github.io/BERTopic/index.html
  

In [ ]:
# %pip install bertopic
# %pip install -q bertopic pandas sentence-transformers scikit-learn stopwordsiso umap-learn hdbscan plotly

In [115]:
import pandas as pd
from bertopic import BERTopic

df = pd.read_csv("../data/css_openalex_26022026.csv")
df = df.dropna(subset=["abstract"])

docs = df["abstract"].tolist()

# # Ou charger des données avec pandas depuis une URL
# data_url = "https://raw.githubusercontent.com/pyshs/CUSO2026/refs/heads/main/data/css_openalex_26022026.csv"
# df_url = pd.read_csv(data_url)
# docs = df_url["abstract"].tolist()

In [116]:
df.shape

(1691, 11)

## Le TM à la hache

### Créer un pipeline et entraîner le modèle

In [117]:
# Le minimum du minimum sans rien affiner :
# (mais en fait plein de choix par défauts qui sont "cachés")
topic_model = BERTopic()
topics, probs = topic_model.fit_transform(docs)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Et explorer les résultats :

In [135]:
# # Les principales fonctions à tester pour avoir un aperçu simple :

# topic_model.get_topic_info()[0:10]
# topic_model.visualize_barchart())
# topic_model.visualize_topics() # Meeh https://github.com/MaartenGr/BERTopic/issues/2269#issuecomment-2607040736
# topic_model.visualize_hierarchy()
# topic_model.visualize_heatmap()
# topic_model.get_document_info(docs)
topic_model.visualize_documents(docs)


## Mais c'est plus compliqué

En réalité, on a une série de briques de traitements, et une série de choix par défauts sont réalisés (et l'on peut aller modifier ces briques) :
- 1 Embeddings
- 2 Dimensionality Reduction
- 3 Clustering
- 4 Tokenizer/vectorizer
- 5 Weighting Scheme
- 6 Representation Tuning

Donc un mille (bon six…) feuilles
![](https://maartengr.github.io/BERTopic/algorithm/default.svg)

Et plein de choix possibles :
![](https://maartengr.github.io/BERTopic/algorithm/modularity.svg)


### On peut donc modifier les étapes

In [136]:
from umap import UMAP
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from bertopic.vectorizers import ClassTfidfTransformer


# Step 1 - Extract embeddings
# all-MiniLM-L6-v2 | paraphrase-multilingual-MiniLM-L12-v2 | all-mpnet-base-v2
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# Step 2 - Reduce dimensionality
umap_model = UMAP(
    n_neighbors=15,  # 15 par défaut
    n_components=5,  # 2 par défaut
    min_dist=0.2,  # 0.1 par défaut
    metric="cosine",  # "euclidean" par défaut
)

# Step 3 - Cluster reduced embeddings
hdbscan_model = HDBSCAN(
    # min_cluster_size=5,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True,
)

# Step 4 - Tokenize topics
vectorizer_model = CountVectorizer(stop_words="english")

# Step 5 - Create topic representation
ctfidf_model = ClassTfidfTransformer()

# Step 6 - (Optional) Fine-tune topic representations with
# a `bertopic.representation` model
representation_model = KeyBERTInspired()

# All steps together
topic_model = BERTopic(
    embedding_model=embedding_model,  # Step 1 - Extract embeddings
    umap_model=umap_model,  # Step 2 - Reduce dimensionality
    hdbscan_model=hdbscan_model,  # Step 3 - Cluster reduced embeddings
    vectorizer_model=vectorizer_model,  # Step 4 - Tokenize topics
    ctfidf_model=ctfidf_model,  # Step 5 - Extract topic words
    representation_model=representation_model,  # Step 6 - (Optional) Fine-tune topic representations
)

topics, probs = topic_model.fit_transform(docs)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# topic_model.get_topic_info()[:20]
# topic_model.visualize_barchart()
# topic_model.visualize_topics()
# topic_model.visualize_hierarchy()
# topic_model.visualize_heatmap()
# topic_model.visualize_documents(docs)
#

,Document,Topic,Name,Representation,Representative_Docs,Top_n_words,Probability,Representative_document
0,A field is emerging that leverages the capacit...,-1,-1_dataset_social_research_data,"[dataset, social, research, data, sciences, re...",[Spatial Data Science Needs Explanations! (And...,dataset - social - research - data - sciences ...,0.000000,False
1,"Data sharing, research ethics, and incentives ...",-1,-1_dataset_social_research_data,"[dataset, social, research, data, sciences, re...",[Spatial Data Science Needs Explanations! (And...,dataset - social - research - data - sciences ...,0.000000,False
2,The integration of social science with compute...,7,7_sociology_social_societies_interdisciplinary,"[sociology, social, societies, interdisciplina...",[This paper presents a comprehensive overview ...,sociology - social - societies - interdiscipli...,0.669190,False
3,Abstract Large language models (LLMs) are capa...,8,8_social_research_conceptualization_researchers,"[social, research, conceptualization, research...",[Large language models (LLMs) have the potenti...,social - research - conceptualization - resear...,0.943694,True
4,Abstract The social sciences investigate human...,7,7_sociology_social_societies_interdisciplinary,"[sociology, social, societies, interdisciplina...",[This paper presents a comprehensive overview ...,sociology - social - societies - interdiscipli...,0.648577,False
...,...,...,...,...,...,...,...,...
1686,The following news item is taken in part from ...,-1,-1_dataset_social_research_data,"[dataset, social, research, data, sciences, re...",[Spatial Data Science Needs Explanations! (And...,dataset - social - research - data - sciences ...,0.000000,False
1687,Author: Khan Tahsin AbrarAffiliation: Independ...,-1,-1_dataset_social_research_data,"[dataset, social, research, data, sciences, re...",[Spatial Data Science Needs Explanations! (And...,dataset - social - research - data - sciences ...,0.000000,False
1688,"Citation (2020), ""Index"", Härtel, C.E.J., Zerb...",35,35_digital_model_graphs_simulated,"[digital, model, graphs, simulated, psychologi...",[<sec> <title>BACKGROUND</title> Emotional sup...,digital - model - graphs - simulated - psychol...,1.000000,True
1689,Publishing academic books in emerging fields p...,48,48_scholarly_interdisciplinary_disciplines_sch...,"[scholarly, interdisciplinary, disciplines, sc...",[With the rising popularity of interdisciplina...,scholarly - interdisciplinary - disciplines - ...,1.000000,True


In [ ]:
# Conserver tout ça :

# Récupérer les infos par document
doc_info = topic_model.get_document_info(docs)

# Fusionner avec le DataFrame originel (par index)
df_merged = pd.concat(
    [df.reset_index(drop=True), doc_info.reset_index(drop=True)], axis=1
)

df_merged.head()

,id,type,primary_location,title,abstract_inverted_index,publication_year,publication_date,open_access,relevance_score,abstract,journal,Document,Topic,Name,Representation,Representative_Docs,Top_n_words,Probability,Representative_document
0,https://openalex.org/W2159397589,article,"{'id': 'doi:10.1126/science.1167742', 'is_oa':...",Computational Social Science,"{'A': [0], 'field': [1], 'is': [2], 'emerging'...",2009.0,2009-02-06,"{'is_oa': True, 'oa_status': 'green', 'oa_url'...",1360.35770,A field is emerging that leverages the capacit...,Science,A field is emerging that leverages the capacit...,-1,-1_dataset_social_research_data,"[dataset, social, research, data, sciences, re...",[Spatial Data Science Needs Explanations! (And...,dataset - social - research - data - sciences ...,0.000000,False
1,https://openalex.org/W3081158114,article,"{'id': 'doi:10.1126/science.aaz8170', 'is_oa':...",Computational social science: Obstacles and op...,"{'Data': [0], 'sharing,': [1], 'research': [2]...",2020.0,2020-08-28,"{'is_oa': True, 'oa_status': 'green', 'oa_url'...",438.53986,"Data sharing, research ethics, and incentives ...",Science,"Data sharing, research ethics, and incentives ...",-1,-1_dataset_social_research_data,"[dataset, social, research, data, sciences, re...",[Spatial Data Science Needs Explanations! (And...,dataset - social - research - data - sciences ...,0.000000,False
2,https://openalex.org/W3022499311,article,{'id': 'doi:10.1146/annurev-soc-121919-054621'...,Computational Social Science and Sociology,"{'The': [0], 'integration': [1], 'of': [2, 16,...",2020.0,2020-04-28,"{'is_oa': True, 'oa_status': 'hybrid', 'oa_url...",413.03424,The integration of social science with compute...,Annual Review of Sociology,The integration of social science with compute...,7,7_sociology_social_societies_interdisciplinary,"[sociology, social, societies, interdisciplina...",[This paper presents a comprehensive overview ...,sociology - social - societies - interdiscipli...,0.669190,False
3,https://openalex.org/W4389636360,article,"{'id': 'doi:10.1162/coli_a_00502', 'is_oa': Tr...",Can Large Language Models Transform Computatio...,"{'Abstract': [0], 'Large': [1], 'language': [2...",2023.0,2023-12-12,"{'is_oa': True, 'oa_status': 'diamond', 'oa_ur...",383.01935,Abstract Large language models (LLMs) are capa...,Computational Linguistics,Abstract Large language models (LLMs) are capa...,8,8_social_research_conceptualization_researchers,"[social, research, conceptualization, research...",[Large language models (LLMs) have the potenti...,social - research - conceptualization - resear...,0.943694,True
4,https://openalex.org/W2790659496,review,"{'id': 'doi:10.1002/wics.95', 'is_oa': False, ...",Computational social science,"{'Abstract': [0], 'The': [1, 52], 'social': [2...",2010.0,2010-05-01,"{'is_oa': False, 'oa_status': 'closed', 'oa_ur...",343.03370,Abstract The social sciences investigate human...,Wiley Interdisciplinary Reviews Computational ...,Abstract The social sciences investigate human...,7,7_sociology_social_societies_interdisciplinary,"[sociology, social, societies, interdisciplina...",[This paper presents a comprehensive overview ...,sociology - social - societies - interdiscipli...,0.648577,False


## Trucs plus avancés
- topic par classe
- évolution des topics dans le temps
- réduction des topics, renommer
- recherche topics proches

## Réduire les topics

In [ ]:
topic_model.reduce_topics(
    docs=docs, nr_topics=10 + 1
)  # Add one to account for the noise

# Retrieve the updated topics and probabilities
topics_reduced, probabilities_reduced = topic_model.topics_, topic_model.probabilities_

In [ ]:
# topic_model.set_topic_labels({
#     0: "AI and Machine Learning",
#     1: "Climate Change and Environment"
# })

In [ ]:
(
    topic_model.visualize_documents(
        docs,
        hide_annotations=True,  # better readability
        topics=[0, 1, 2, 3],  # Select topics to highlight
        # height = 300, # Adjust the height of the plot
        # width = 800 # Adjust the width of the plot
    )
)

In [ ]:
topic_model.visualize_barchart(
    n_words=10,  # Select the number of words to display per topic
    # topics = [0,1,2,3,4], # Select specific topics to display
    # top_n_topics = 6, # Select the first n topics to display
    # height = 300, # Adjust the height of the plot
    # width = 800 # Adjust the width of the plot
)

## À garder en tête

**Choix embeddings**
- fenetre contexte
- adapté à la langue
- adapté à la tache
- les précalculer pour pas avoir à relancer le calcul à chaque fois… 

**reproductibilité**
- set seed

**Évaluer**
- mais surtout éval qualitative

**Itérer**
- tester des trucs

In [147]:
## à intégrer depuis axel :

Save your instance locally

For reproducibility purposes, BERTopic lets you save the BERTopic object you created with the save17 method. Two parameters of importance:

    serialization (str): must be "safetensors", "pickle" or "pytorch". We recommend using "safetensors" or the "pytorch" format as they are broadly used in machine learning and recommended by the BERTopic documentation.
    save_ctfidf (bool): whether to save the vectorizer configuration or not. This is the heaviest bit (see table below).


Precompute your embeddings

Pre-computing the embeddings is a good practice as it will prevent from computing them at each run, but also because it allows you to use a broader spectrum of embedding models. This comes handy when you want to test different parameters of clustering and cluster representation. Moreover, saving BERTopics models does not save the embeddings, so it is good practice to manage them separately.

To embed our documents, we use the datasets objects to manage the data and the sentence-bert (SBERT) library to embed the documents. The process is very straightforward, you need to open your file and preprocess your texts. Then after loading the model

In [ ]:
from datasets import Dataset
from gc import collect as gc_collect
from sentence_transformers import SentenceTransformer
from torch.cuda import is_available as cuda_available
from torch.cuda import synchronize, ipc_collect, empty_cache

ds = Dataset.load_from_disk("...")
# implement your preprocess and open functions
texts: list[str] = preprocess(ds["texts"])
# Use GPU if you have one
device = "cuda" if cuda_available() else "cpu"

sbert_model = SentenceTransformer(model_name, device=device, trust_remote_code=True)

sbert_model.max_seq_length = min(
    sbert_model.max_seq_length,
    np.inf,  # Replace with desired window size
)

try:
    embeddings = sbert_model.encode(
        texts, device=str(device), normalize_embeddings=True, show_progress_bar=True
    )
    ds = ds.add_column("embedding", list(embeddings))
    ds.save_to_disk("embeddings")
except Exception:
    print(Exception)
finally:
    # Make sure to clean your GPU
    del sbert_model, ds
    empty_cache()
    if cuda_available():
        synchronize()
        ipc_collect()
    gc_collect()

In [ ]:
ds = load_from_disk("path/to/file")
docs = np.array(ds[f"texts"])  # Number of documents: 6500
embeddings = np.array(ds["embedding"])  # shape: (6500, 768)

Force deterministic behaviour

The BERTopic pipeline is deterministic apart from the UMAP component. To force a deterministic behaviour:

topic_model = BERTopic(
    ...
    umap_model= UMAP(
        ...
        random_state=RANDOM_SEED
    )
)